# Bluesky Data Science
## Part 00 - Introduction

November 2024 (revised November 2026)
Copyright and distributed under an [MIT License](https://opensource.org/licenses/MIT)

*An [R Markdown counterpart](../RMarkdown/Part%2000%20-%20Introduction.Rmd) of this notebook, using the [bskyr](https://christophertkenny.com/bskyr/) package, lives in the parallel `RMarkdown/` directory.*

## Background

> **tl;dr**: This is a series of notebooks introducing functionality for doing common data science tasks using the ATProto protocol powering Bluesky, a post-Twitter microblogging platform.

### RIP Twitter

After Twitter's acquisition by Elon Musk in October 2022, the platform rapidly deteriorated through active mismanagement by dismissing the teams dedicated to trust and safety, policy, and research. Bots, spam, and extremist content have proliferated and Musk's active campaigning on behalf of Donald Trump's 2024 presidential campaign has further disillusioned users.

Access to Twitter data via its API was also enclosed behind an exorbitantly priced paywall, effectively shutting out researchers, journalists, and students. Given the increasingly precarity of Twitter's API access, I no longer recommend that researchers build research projects around Twitter. 

### Bluesky
Alternative microblogging services like [Bluesky](https://bsky.app) and [Mastodon](https://joinmastodon.org/) have emerged in this aftermath. Bluesky, in particular, appears to have attracted a critical mass of regular and influential users as well as persuasive features for safety, on-boarding new users, and data access. You don't need an account to follow most of this series: Bluesky's public data can be read without logging in. An account, used through an *app password* (explained below), unlocks the parts that are private to you or that write to the network: your timeline, notifications, search, and posting.

Bluesky runs on a decentralized social networking software called the "authenticated transfer protocol" or "ATProto". More details about ATProto:

* [Documentation](https://docs.bsky.app/docs/advanced-guides/atproto)
* [Github repo](https://github.com/bluesky-social/atproto)
* [Wikipedia article](https://en.wikipedia.org/wiki/AT_Protocol)

As data scientists we want to responsibly retrieve data from the social network to understand the behavior of its users. Instead of using web scraping techniques to retrieve this data through a web browser, we will leverage the application programming interfaces (APIs) provided by ATProto to get (and post) data about posts, users, timelines, *etc*. We are going to leverage the ATProto Python software development kit (SDK) developed by Ilya Siamionau:

* [Documentation](https://atproto.blue/en/latest/)
* [Github repo](https://github.com/MarshalX/atproto)

We are going to build on many of the [examples](https://github.com/MarshalX/atproto/tree/main/examples) published in the ATProto Python SDK.

### Code
This is a Jupyter Notebook, a framework that allows developers and researchers to write code and documentation inside a file that can also execute code and embed tables and figures. These notebooks run in a Python *virtual environment* with the exact package versions pinned in the repository's `requirements.txt` (see Setup below). Please consult any of the many tutorials about using Jupyter Notebooks if you are not already familiar:

* [Documentation](https://docs.jupyter.org/en/latest/)
* [Dataquest tutorial](https://www.dataquest.io/blog/jupyter-notebook-tutorial/)
* [Datacamp tutorial](https://www.datacamp.com/tutorial/tutorial-jupyter-notebook)
* [Codecademy](https://www.codecademy.com/article/how-to-use-jupyter-notebooks)

The rest of the notebooks will assume familiarity with basic Python data structures (lists, dictionaries), flow control and exception handling (`if`, `for`, `while`, `try`), and objects (methods, attributes).


### Learning objectives

By the end of this notebook you will be able to:

1. Explain what Bluesky and the AT Protocol are, and the difference between reading public data anonymously and logging in to an account.
2. Create an **app password** and store your credentials safely in an `atproto.json` file.
3. Connect with `get_client()` and check whether you are authenticated.
4. Retrieve profiles, author feeds, follows, and followers, and read the objects they return: **handles**, **DIDs**, posts, and records.
5. Use a **cursor** to fetch the next page of results.
6. Describe how to write to the network (posts, likes, images with alt text), and why these tutorials never do it automatically.

### Setup

Each project should get its own **virtual environment**: a private folder of Python packages, so this series' pinned versions can't collide with other projects (or with packages your operating system installed). From the repository root, in a terminal:

```bash
python3 -m venv .venv                 # create the environment (once)
source .venv/bin/activate             # activate it (every new terminal; on Windows: .venv\Scripts\activate)
pip install -r requirements.txt       # install the pinned packages (once)
jupyter notebook                      # launch Jupyter from inside the environment
```

(Prefer conda? `conda create -n bluesky python=3.11 && conda activate bluesky && pip install -r requirements.txt` works too.)

**You should only need to do this once.** The next cell checks that the key packages are installed and shows their versions, which is worth noting in any write-up — the `atproto` package is pre-1.0, and its maintainers warn that compatibility between versions is not guaranteed.

In [1]:
from importlib.metadata import version

for package in ['atproto', 'pandas']:
    print(f'{package:<8} {version(package)}')

atproto  0.0.73
pandas   3.1.0


## Client

There are two ways to talk to Bluesky, and it helps to know which one you are using:

* **Reading public data, without logging in.** Bluesky runs a public **AppView** at `https://public.api.bsky.app` — the service that assembles the network's raw records into the profiles, posts, and feeds you see in the app. Anyone can ask it for public profiles, posts, follows, and followers. No account needed.
* **Logging in to an account.** Some things only make sense relative to *you*: your timeline, your notifications, search, and anything that writes to the network (posting, liking). For these you log in to your account's **PDS** (Personal Data Server: the server that stores your account's data), which checks who you are.

### App passwords: never use your real password in code

To log in from code, create an **app password**: a separate, randomly generated password that works only for apps like this one. In the Bluesky app, go to **Settings → Privacy and security → App passwords**, choose *Add App Password*, give it a name like "data science notebooks", and copy the password it shows you (it is shown only once). An app password can't change your account's email, password, or other security settings, and you can revoke it at any time without affecting anything else. Never type your main account password into a notebook: notebooks get shared, committed to GitHub, and screenshotted.

Save your handle and app password in a file called `atproto.json` in the **repository root** (the folder that contains `README.md`):

```json
{"handle": "your-handle.bsky.social", "password": "xxxx-xxxx-xxxx-xxxx"}
```

The repository's `.gitignore` already lists `atproto.json`, so git won't commit it by accident. Never paste credentials into a notebook cell.

### Connecting

Every notebook in this series connects the same way, with `get_client()` from the small helper file [`bsky_client.py`](bsky_client.py). If it finds an `atproto.json` it logs in; if not, it returns a client pointed at the public AppView. `is_authenticated()` tells you which one you got. Open `bsky_client.py` and read it — it is about 30 lines.

In [2]:
from bsky_client import get_client, is_authenticated

client = get_client()
is_authenticated(client)

No atproto.json found: using the unauthenticated public AppView.
Read endpoints work; search and write endpoints require logging in.


False

If the output above says `False`, you are reading anonymously, and every cell in this notebook that needs an account will explain itself and skip. That's fine: most of this notebook, and most of the series, works either way.

## Profile

`get_profile()` ([`app.bsky.actor.getProfile`](https://endpoints.bsky.app/#bluesky-app/tag/appbskyactor/GET/xrpc/app.bsky.actor.getProfile)) retrieves an account's public profile. We'll use The New York Times (`nytimes.com`), one of this series' two standing example accounts: a large institutional account, whose posting is professional output rather than a private person's life.

Two identifiers appear in every profile, and the difference matters for research:

* The **handle** (`nytimes.com`) is the human-readable name. Handles are domain names, and an account can change its handle at any time.
* The **DID** (Decentralized Identifier, `did:plc:...`) is the account's permanent, machine-readable ID. It never changes, even when the handle does. When you store data about accounts, key it on DIDs.

In [3]:
profile = client.get_profile('nytimes.com')

print('Handle:      ', profile.handle)
print('DID:         ', profile.did)
print('Display name:', profile.display_name)
print('Followers:   ', f'{profile.followers_count:,}')
print('Follows:     ', f'{profile.follows_count:,}')
print('Posts:       ', f'{profile.posts_count:,}')
print('Created:     ', profile.created_at)

Handle:       nytimes.com
DID:          did:plc:eclio37ymobqex2ncko63h4r
Display name: The New York Times
Followers:    1,355,663
Follows:      123
Posts:        44,906
Created:      2023-06-05T18:50:31.498Z


These counts are **live**: run the cell tomorrow and the follower count will have moved. Any number you collect from an API is a snapshot, which is why later parts always record *when* they collected their data.

The SDK returns a *model* object (a [Pydantic](https://docs.pydantic.dev/) model — a Python class with typed fields), not a plain dictionary. Its fields are attributes, like `profile.followers_count` above. `.model_dump()` converts the model into an ordinary dictionary, which is the easiest way to see every field it has:

In [4]:
sorted(profile.model_dump().keys())

['associated',
 'avatar',
 'banner',
 'created_at',
 'debug',
 'description',
 'did',
 'display_name',
 'followers_count',
 'follows_count',
 'handle',
 'indexed_at',
 'joined_via_starter_pack',
 'labels',
 'links',
 'pinned_post',
 'posts_count',
 'pronouns',
 'py_type',
 'status',
 'verification',
 'viewer',
 'website']

### Your own profile

If you're logged in, `client.me` holds your own handle and DID, so you can look yourself up the same way:

In [5]:
if is_authenticated(client):
    me = client.get_profile(client.me.handle)
    print(f'You are {me.handle}: {me.followers_count:,} followers, {me.posts_count:,} posts')
else:
    print('Not logged in, so there is no "own profile" to show. With an atproto.json, this cell')
    print('would print your handle, follower count, and post count.')

Not logged in, so there is no "own profile" to show. With an atproto.json, this cell
would print your handle, follower count, and post count.


### Retrieving other users' profiles

Any public account works. `bsky.app`, the platform's official account, is the series' other standing example:

In [6]:
bsky_app = client.get_profile('bsky.app')

print(f'{bsky_app.handle}: {bsky_app.followers_count:,} followers, '
      f'follows {bsky_app.follows_count:,} accounts, {bsky_app.posts_count:,} posts')

bsky.app: 35,496,872 followers, follows 34 accounts, 907 posts


Note the asymmetry: millions of people follow the official account, and it follows almost no one. Audience size and network ties are different things — a distinction Part 03 builds a whole network analysis on.

## Posts

`get_author_feed()` ([`app.bsky.feed.getAuthorFeed`](https://endpoints.bsky.app/#bluesky-app/tag/appbskyfeed/GET/xrpc/app.bsky.feed.getAuthorFeed)) returns an account's recent posts, newest first, up to 100 per request.

What comes back is a list of **feed items** in `.feed`, and each item wraps three layers. They take a moment to learn, and you'll use them in every part:

* `item.post` — the **post view**: the post as the AppView presents it, with its `author`, its engagement counters (`like_count`, `repost_count`, `reply_count`), and identifiers (`uri`, `cid`).
* `item.post.record` — the **record**: exactly what the author's app wrote to their repository, including `text`, `created_at`, and `langs` (declared languages).
* `item.reason` — set only when the item is a **repost**: the account is amplifying someone else's post rather than writing its own.

In [7]:
feed = client.get_author_feed(actor='nytimes.com', limit=100)

len(feed.feed)

100

In [8]:
item = feed.feed[0]
post = item.post

print('Repost?   ', item.reason is not None)
print('Author:   ', post.author.handle)
print('Written:  ', post.record.created_at)
print('Text:     ', post.record.text[:200])
print('Likes:    ', post.like_count, '| Reposts:', post.repost_count, '| Replies:', post.reply_count)
print('URI:      ', post.uri)

Repost?    False
Author:    nytimes.com
Written:   2026-11-02T19:40:04.190869Z
Text:      The collapse of the relationship between Representative Haley Stevens and Dr. Abdul El-Sayed has created a potentially costly drama for Democrats in a race that holds Republicans’ best chance of flipp
Likes:     4 | Reposts: 1 | Replies: 16
URI:       at://did:plc:eclio37ymobqex2ncko63h4r/app.bsky.feed.post/3mze3bztkiq2s


The **URI** (`at://did:plc:.../app.bsky.feed.post/...`) is the post's permanent address on the AT Protocol: the author's DID, the record type, and a key for this particular record. Together with the **CID** (a hash of the post's exact content), it's how you point at a post — for example, to like or reply to it.

## Timeline

Your **timeline** is the feed of posts from accounts *you* follow. It is defined relative to the logged-in account, so there is no anonymous version of it:

In [9]:
if is_authenticated(client):
    timeline = client.get_timeline(algorithm='reverse-chronological', limit=25)
    print(f'{len(timeline.feed)} posts in your timeline; the first five authors:')
    for item in timeline.feed[:5]:
        print('  ', item.post.author.handle)
else:
    print('Requires authentication: the timeline is "posts from accounts I follow",')
    print('so the server needs to know who "I" am.')

Requires authentication: the timeline is "posts from accounts I follow",
so the server needs to know who "I" am.


## Relationships

### Follows

`get_follows()` ([`app.bsky.graph.getFollows`](https://endpoints.bsky.app/#bluesky-app/tag/appbskygraph/GET/xrpc/app.bsky.graph.getFollows)) returns the accounts an account follows, at most 100 per request.

When there are more results than fit in one response, the API returns a **cursor** with the page: an opaque bookmark marking where the next page starts. You pass it back to get the next page. This is called **pagination**, and every list endpoint in the series works this way.

In [10]:
follows = client.get_follows(actor='nytimes.com', limit=100)

print('Accounts on page 1:', len(follows.follows))
print('Cursor for page 2: ', follows.cursor)

Accounts on page 1: 100
Cursor for page 2:  3limsedin7r22


Pass that cursor back to get the next page, and check that the two pages don't overlap:

In [11]:
next_follows = client.get_follows(actor='nytimes.com', limit=100, cursor=follows.cursor)

page1 = {account.did for account in follows.follows}
page2 = {account.did for account in next_follows.follows}

print('Accounts on page 2:   ', len(page2))
print('Overlap with page 1:  ', len(page1 & page2))
print('Cursor for page 3:    ', next_follows.cursor)

Accounts on page 2:    23
Overlap with page 1:   0
Cursor for page 3:     None


Zero overlap: page 2 picks up exactly where page 1 left off. When an account follows few enough people, the second page comes back short, and a cursor of `None` means there is nothing left to fetch. Part 01 turns this into a loop that collects a complete list, and shows how to budget the requests it needs.

### Followers

`get_followers()` is the mirror image: the accounts that follow an account. Followers are mostly private individuals who never chose to be in your dataset, so rather than printing someone's profile, we look at what *kind* of information each record contains:

In [12]:
followers = client.get_followers(actor='nytimes.com', limit=100)

print('Followers on this page:', len(followers.followers))
print('Fields in each follower record:')
print(sorted(followers.followers[0].model_dump().keys()))

Followers on this page: 95
Fields in each follower record:


['associated', 'avatar', 'created_at', 'debug', 'description', 'did', 'display_name', 'handle', 'indexed_at', 'labels', 'pronouns', 'py_type', 'status', 'verification', 'viewer']


Showing the structure instead of a stranger's profile is a habit worth keeping. You can design an analysis without looking at any individual's data, and when you do need records, store and report them in aggregate. The Ethics section below says more.

## Posting content

Everything above **reads** the network. The SDK can also **write**: post, like, reply, and attach images. Writing changes the public record under your name, so this series never does it automatically: the examples below are shown as code, not executable cells. All of them need you to be logged in (with an app password), and they act as *your* account.

**Writing a post**

```python
client.send_post('Posting from a Jupyter notebook in my data science class!')
```

**Liking a post.** A like points at a post by its URI and CID, both of which every post view carries (see the Posts section above):

```python
client.like(uri=post.uri, cid=post.cid)
```

**Posting an image, with alt text.** `image_alt` is the image's **alt text**, a written description that screen readers speak aloud to blind and low-vision people. Always write one. Part 07 measures how often accounts do.

```python
with open('my_image.jpg', 'rb') as f:
    image_data = f.read()

client.send_image(
    text='Now we are sending images from a notebook!',
    image=image_data,
    image_alt='A short description of what the image shows',
)
```

Run these deliberately, by hand, when you mean to post. Part 10 builds a bot that posts automatically, along with the disclosure and rate limits that a bot needs.

## Ethics and limitations

**Public is not the same as consented.** Everything this notebook read is public: anyone with a browser could see it. That doesn't mean the people behind it expect to appear in a dataset. Most Bluesky users have never read the protocol documentation, and research on Twitter found that users are often surprised, and sometimes upset, to learn their posts are used in research (Fiesler & Proferes 2018). This series studies institutional accounts by default, prefers aggregates to individual records, and treats ethics as a design decision made *before* collecting — a theme Part 01 develops in depth.

**Protect your credentials.** An app password can post as you. Keep it in `atproto.json` (never in a notebook), don't share it, and revoke it if it leaks.

**Limitations.** Everything here is a live snapshot: counts and feeds change by the minute, posts get deleted, and handles change (DIDs don't). The AppView shows the network as Bluesky's own service assembles it. And Bluesky's users are not "the public": the platform skews toward early adopters and people who left Twitter, so nothing measured here describes society at large.

## Exercises

1. **Another account.** Retrieve the profile of a news organization, university, or government agency you are interested in. Compare its follower and follow counts with `nytimes.com`'s. What does the ratio suggest about how the account uses Bluesky?
2. **Handles versus DIDs.** Look up `bsky.app` by its DID instead of its handle (`client.get_profile(bsky_app.did)`). Do you get the same profile? Why is this useful for research that runs over months?
3. **Reposts in a feed.** Count how many of the 100 feed items in `feed.feed` are reposts (`item.reason is not None`). What share of the account's recent activity is amplification rather than original posting?
4. **Follow the cursor.** Write a short loop that keeps calling `get_follows` with the latest cursor until the cursor is `None`, and count the accounts it collected. Compare your total with `profile.follows_count`.
5. **Explore the fields.** Pick one field from the `sorted(profile.model_dump().keys())` list that you didn't recognize, look it up in the [API reference](https://endpoints.bsky.app/#bluesky-app/tag/appbskyactor/GET/xrpc/app.bsky.actor.getProfile), and write one sentence on how a researcher might use it.

## Research project ideas

**1. Institutional adoption of Bluesky.** *Question:* how have newsrooms, universities, or government agencies adopted Bluesky since 2024? *Data:* the profiles and recent posting of a defined population of institutional accounts (for example, every newspaper on a public list), collected with the endpoints from this notebook. *Method:* describe account ages, posting volumes, and follower counts, then compare how sectors use the platform. *Expected challenge:* building the population list — finding *which* accounts exist is the hard part, and Part 02 introduces ways to do it.

**2. Audience versus attention.** *Question:* do accounts with bigger audiences get proportionally more engagement per post? *Data:* follower counts and the most recent 100 posts' like counts for a few dozen public institutional accounts. *Method:* plot median likes per post against follower count on log scales (Part 04 explains why the log scale matters). *Expected challenge:* engagement depends heavily on how often an account posts and what it posts, so a fair comparison needs to control for posting volume.

## References

### Scholarship

* boyd, d., & Crawford, K. (2012). Critical questions for big data. *Information, Communication & Society*, 15(5), 662–679.
* Fiesler, C., & Proferes, N. (2018). "Participant" perceptions of Twitter research ethics. *Social Media + Society*, 4(1).
* Freelon, D. (2018). Computational research in the post-API age. *Political Communication*, 35(4), 665–668.
* Kleppmann, M., et al. (2024). Bluesky and the AT Protocol: Usable decentralized social media. *arXiv:2402.03239*.

### Documentation

* Bluesky HTTP API reference: <https://endpoints.bsky.app/>
* AT Protocol documentation and specifications (DIDs, handles, records): <https://atproto.com/docs>
* ATProto Python SDK documentation: <https://atproto.blue/>
* Creating app passwords: Settings → Privacy and security → App passwords in the Bluesky app